## 为直井建立局部时深关系并输出 checkshots 文件


In [ ]:
import sys
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# 从 notebooks/ 回到仓库根目录
repo_root = Path.cwd().resolve().parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.export import export_vertical_tdt_to_petrel_checkshots
from cup.petrel.load import (
    import_interpretation_petrel,
    import_well_heads_petrel,
    import_well_tops_petrel,
    load_vp_rho_logset_from_las,
)
from cup.seismic.process import interpolate_interpretation_surface
from cup.seismic.survey import open_survey
from cup.well.process import clip_logsets_by_well_tops

#  from wtie.optimize import similarity, tie
# from wtie.optimize.wavelet import compute_expected_wavelet
from wtie.processing import grid
# from wtie.utils import viz
# from wtie.utils.datasets import tutorial
# from wtie.utils.datasets.utils import InputSet

## 定义


In [ ]:
# 定义井名
well_name = "NW7"

data_root = repo_root / "data"
well_heads_file = data_root / "raw" / "well_heads"
well_tops_file = data_root / "raw" / "well_tops_new"
las_dir = data_root / "vertical_well_las_target_clear"

top_surface_name = "BVE100_TOP"
base_surface_name = "ITP_BOT"

seismic_file = data_root / "raw" / "mero se 0116_1ms_new_84_coord.Sgy"
seismic_interpretation_dir = data_root / "interpre_time"

seismic_iline = 5
seismic_xline = 21
seismic_istep = 1
seismic_xstep = 4

top_interpretation_horizon_name = "bve_top_t"
top_interpretation_horizon_file = seismic_interpretation_dir / f"{top_interpretation_horizon_name}"
interpretation_offset = -0.006  # s

seismic_ctx = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": seismic_iline,
        "xline": seismic_xline,
        "istep": seismic_istep,
        "xstep": seismic_xstep,
    },
)

## 导入井头、井分层文件


In [ ]:
well_heads_df = import_well_heads_petrel(well_heads_file)
well_tops_df = import_well_tops_petrel(well_tops_file)

print("well_heads shape:", well_heads_df.shape)
print("well_tops shape:", well_tops_df.shape)

display(well_heads_df.head())
display(well_tops_df.head())

## 导入井曲线并裁剪到目标层


In [ ]:
las_files = sorted(las_dir.glob("*.las"))
print(f"LAS files found: {len(las_files)}")
for p in las_files:
    print(" -", p.name)

logsets = {}
failed = []

for las_path in las_files:
    try:
        logset = load_vp_rho_logset_from_las(las_path)

        # 以文件名（不含后缀）作为井名键
        logsets[las_path.stem] = logset

    except Exception as e:
        failed.append((las_path.name, str(e)))

print(f"\n成功构建 LogSet 数量: {len(logsets)}")
print(f"失败数量: {len(failed)}")
if failed:
    for name, msg in failed:
        print(f"[FAILED] {name}: {msg}")

clip_result = clip_logsets_by_well_tops(
    well_tops_df=well_tops_df,
    logsets=logsets,
    top_surface_name=top_surface_name,
    base_surface_name=base_surface_name,
)
clipped_logsets = clip_result["processed_logsets"]
print(f"层位回退次数: {clip_result['surface_fallback_count']}")

print(f"裁剪层位区间: {top_surface_name} -> {base_surface_name}")
print(f"裁剪后 LogSet 数量: {len(clipped_logsets)}")

## 导入地震和地震解释层位（并插值）


In [ ]:
interpretation_df = import_interpretation_petrel(top_interpretation_horizon_file)

geometry = seismic_ctx.query_geometry(domain="time")

interpolate_interpretation_df = interpolate_interpretation_surface(
    interpretation_df=interpretation_df,
    geometry=geometry,
    outlier_threshold=0.02,
    min_neighbor_count=2,
    keep_nan=True,
)

# 插值后层位应全部有效，否则直接报错
assert interpolate_interpretation_df["interpretation"].notna().all(), (
    "插值后 interpretation 仍包含 NaN，视为插值异常，请检查输入层位或插值参数"
)

## 提取井旁道


In [ ]:
row = well_heads_df.loc[well_heads_df["Name"] == well_name]
assert not row.empty, f"well_name={well_name} 在井头表中不存在"
assert row.shape[0] == 1, f"well_name={well_name} 在井头表中有重复记录: {row.shape[0]} 条"

kb = float(row.iloc[0]["Well datum value"])
wellX = float(row.iloc[0]["Surface X"])
wellY = float(row.iloc[0]["Surface Y"])

print(f"well_name = {well_name}")
print(f"kb = {kb:.3f} m")
print(f"wellX = {wellX:.3f} m")
print(f"wellY = {wellY:.3f} m")

seismic_trace = seismic_ctx.import_seismic_at_well(
    well_x=wellX,
    well_y=wellY,
    domain="time",
)

## 构建局部时深关系表


In [ ]:
assert well_name in clipped_logsets, (
    f"well_name={well_name} 不在裁剪后的 logsets 键中。可用键示例: {list(clipped_logsets.keys())[:10]}"
)

vp_local = clipped_logsets[well_name].Logs["Vp"]
wp_local = grid.WellPath(vp_local.basis, kb)

# 井位坐标 -> 浮点道号
il_float, xl_float = seismic_ctx.coord_to_line(x=wellX, y=wellY)

# 关键修正：邻点必须落在有效网格线（min + k*step）上
il_min = float(geometry["inline_min"])
xl_min = float(geometry["xline_min"])
il_step = float(geometry["inline_step"])
xl_step = float(geometry["xline_step"])
assert il_step > 0 and xl_step > 0, "geometry 步长必须为正"

k_il = (il_float - il_min) / il_step
k_xl = (xl_float - xl_min) / xl_step

k_il0 = int(np.floor(k_il))
k_il1 = int(np.ceil(k_il))
k_xl0 = int(np.floor(k_xl))
k_xl1 = int(np.ceil(k_xl))

il0 = int(round(il_min + k_il0 * il_step))
il1 = int(round(il_min + k_il1 * il_step))
xl0 = int(round(xl_min + k_xl0 * xl_step))
xl1 = int(round(xl_min + k_xl1 * xl_step))

# 边界点时 floor==ceil，权重会自动退化到线性/点值
wi = k_il - k_il0
wj = k_xl - k_xl0


def _get_interp_value(il, xl):
    row = interpolate_interpretation_df.loc[
        (interpolate_interpretation_df["inline"] == il) & (interpolate_interpretation_df["xline"] == xl)
    ]
    assert not row.empty, f"解释层位缺少网格点: inline={il}, xline={xl}"
    val = float(row.iloc[0]["interpretation"])
    assert np.isfinite(val), f"解释层位无效值: inline={il}, xline={xl}, val={val}"
    return val


t00 = _get_interp_value(il0, xl0)
t01 = _get_interp_value(il0, xl1)
t10 = _get_interp_value(il1, xl0)
t11 = _get_interp_value(il1, xl1)

interp_value = (1.0 - wi) * (1.0 - wj) * t00 + (1.0 - wi) * wj * t01 + wi * (1.0 - wj) * t10 + wi * wj * t11

sample_unit = str(geometry.get("sample_unit", "")).lower()
if sample_unit == "ms":
    interp_s = interp_value / 1000.0
elif sample_unit == "s":
    interp_s = interp_value
else:
    raise ValueError(f"Unsupported sample_unit for time domain: {sample_unit}")

origin_s = max(0.0, interp_s + float(interpretation_offset))

local_tdt = grid.TimeDepthTable.get_tvdss_twt_relation_from_vp(
    Vp=vp_local,
    wp=wp_local,
    origin=origin_s,
)

# checkshots_file = data_root / "TD" / f"{well_name}_local_tdt.checkshots.txt"
# export_vertical_tdt_to_petrel_checkshots(
#     output_file=checkshots_file,
#     tdt=local_tdt,
#     well_name=well_name,
#     kb=kb,
#     x=wellX,
#     y=wellY,
# )

# print(f"checkshots 已导出: {checkshots_file}")
print(f"coord_to_line 浮点道号: inline={il_float:.3f}, xline={xl_float:.3f}")
print(f"四邻点(按步长对齐): ({il0},{xl0}), ({il0},{xl1}), ({il1},{xl0}), ({il1},{xl1})")
print(f"层位时间(双线性插值): {interp_s * 1000.0:.3f} ms ({interp_s:.6f} s)")
print(f"层位时间(加上 offset={interpretation_offset:.3f} s): {origin_s * 1000.0:.3f} ms")

# 小图检查局部时深关系（twt 期望随深度单调增加）
fig, ax = plt.subplots(figsize=(4, 5))
ax.plot(local_tdt.twt * 1000.0, local_tdt.tvdss, lw=1.2)
ax.invert_yaxis()
ax.set_xlabel("TWT (ms)")
ax.set_ylabel("TVDSS (m)")
ax.set_title(f"Local Time-Depth: {well_name}")
ax.grid(True, alpha=0.3)
plt.show()